# Thu thập và khám phá dữ liệu TMDB + IMDb

**Bản làm việc ngày 15/11/2026.** Đầu vào là CSV nhóm tải ngày 05/11/2026 theo README.
Mục tiêu: kiểm tra cấu trúc, chất lượng và các mẫu hình sơ bộ trước khi chốt câu hỏi.
Các thống kê raw đếm **bản ghi**, chưa mặc định mỗi dòng là một phim duy nhất.

Chọn kernel **Python (movies-dataset)** rồi **Restart Kernel and Run All Cells**.
Notebook chạy từ thư mục gốc hoặc `notebooks/`, không tải dữ liệu khi chạy và không ghi đè CSV.
Các phép chuyển kiểu, tách thể loại và lọc mẫu chỉ phục vụ kiểm tra trong bộ nhớ.

Codex hỗ trợ mã kiểm tra và ghi chú kỹ thuật; xem [nhật ký LLM](../LLM_USAGE.md).
Theo mục 6 của [đề bài](../doc/CSC17104_2026_Final_Project.pdf), nhóm cần hiểu từng dòng mã,
tự viết phần phân tích, câu trả lời và reflection cho bài nộp.

## 1. Nguồn và bối cảnh

- **Chủ đề:** metadata tác phẩm, ngân sách/doanh thu được ghi nhận và đánh giá khán giả từ TMDB và IMDb.
- **Bản tổng hợp:** [TMDB + IMDB Merged Movies Dataset](https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset),
  do **Tejas Garg (`ggtejas`)** công bố. [Metadata Kaggle API](https://www.kaggle.com/api/v1/datasets/view/ggtejas/tmdb-imdb-merged-movies-dataset)
  được kiểm tra ngày **15/11/2026**: giấy phép **CC BY-NC-SA 4.0**, phiên bản hiện tại 587,
  cập nhật `2026-11-14T13:10:18.11Z`. Đây là thông tin trang hiện tại, **không phải phiên bản đã xác minh của CSV cục bộ**.
- **Ngày công bố lần đầu:** chưa xác minh được từ metadata trả về; không dùng ngày cập nhật thay cho ngày công bố.
  **Ngày nhóm tải:** 05/11/2026 theo README, khác ngày file được sao chép vào máy này.
- **Nguồn gốc:** mô tả của tác giả dẫn [TMDB dataset của asaniczka](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies)
  và [IMDb Non-Commercial Datasets](https://developer.imdb.com/non-commercial-datasets/).
  Nhóm sử dụng CSV tổng hợp có sẵn, không trực tiếp thu thập lại toàn bộ TMDB/IMDb.
- **Cách kết hợp:** metadata TMDB được bổ sung đạo diễn, biên kịch, diễn viên và đánh giá IMDb.
  CSV có `id` và `tconst`. Mô tả nguồn chưa xác nhận thuật toán ghép, loại join, quy tắc khử trùng
  hay thời điểm chụp từng bảng. Không khẳng định đã join một-một hoặc ghép đúng chỉ vì có hai mã.
- **Giấy phép:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) của bản tổng hợp;
  ghi nguồn và các thay đổi, sử dụng phi thương mại, tuân thủ điều kiện chia sẻ tương tự.
  Đối chiếu thêm điều kiện của [IMDb](https://developer.imdb.com/non-commercial-datasets/)
  và [TMDB](https://developer.themoviedb.org/docs/faq); nhãn của bản tổng hợp không thay thế điều kiện nguồn.
- **Phạm vi và giới hạn:** các tác phẩm có mặt trong CSV, không phải mẫu ngẫu nhiên của toàn bộ điện ảnh.
  Không có `titleType` để xác định chắc chắn phim điện ảnh/phim ngắn/truyền hình.
  Thời gian phát hành không phải thời gian thu thập. Đánh giá và độ phổ biến là ảnh chụp, có thể khác thời điểm.
  Thiếu tiền tệ theo từng dòng và cách chuẩn hóa tài chính, không tự khẳng định mọi số tiền đã quy đổi USD
  hoặc điều chỉnh lạm phát. Doanh thu/ngân sách không đồng nghĩa lợi nhuận.
- **Lý do chọn theo README:** chủ đề gần gũi; có tài chính, thuộc tính đa nhãn và điểm từ hai nền tảng,
  phù hợp tìm hiểu các khác biệt theo nhóm và theo thời gian. Nhóm bổ sung động cơ cá nhân bằng lời của mình.

Tài liệu IMDb định nghĩa `averageRating` là điểm trung bình có trọng số và `numVotes` là số lượt chấm;
`tconst` là mã tác phẩm. Metadata nguồn là mô tả của người công bố, không chứng minh chất lượng từng bản ghi.

In [ ]:
from pathlib import Path
import csv
import hashlib
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'environment.yml').is_file() and (p / 'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook từ thư mục repository hoặc notebooks/.')
RAW_PATH = ROOT / 'data/raw/TMDB  IMDB Movies Dataset.csv'
if not RAW_PATH.is_file():
    raise FileNotFoundError(f'Tải CSV theo README và đặt tại: {RAW_PATH}')
DOWNLOAD_DATE = pd.Timestamp('2026-11-05')  # Theo README; không phải ngày công bố.
RANDOM_STATE = 42
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_columns', 12)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 11, 'axes.labelsize': 10})
display(pd.Series({'Python': platform.python_version(), 'pandas': pd.__version__,
                   'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
                   'seaborn': sns.__version__, 'executable': sys.executable}, name='Môi trường'))

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

raw_sha256 = file_sha256(RAW_PATH)
print('SHA-256:', raw_sha256)
# Giữ nguyên chuỗi như NA/N/A/NULL nếu có; chỉ ô trống và ký hiệu IMDb \N là thiếu.
movies = pd.read_csv(RAW_PATH, low_memory=False, keep_default_na=False, na_values=['', r'\N'])
n = len(movies)
display(pd.Series({'dòng': n, 'cột': movies.shape[1], 'byte': RAW_PATH.stat().st_size,
                   'SHA-256': raw_sha256, 'ngày tải theo README': str(DOWNLOAD_DATE.date())}, name='CSV cục bộ'))
display(movies[['id', 'tconst', 'title', 'release_date', 'budget', 'revenue']].head())

## 2. Cấu trúc, dòng rỗng và định danh

Đếm bản ghi bằng bộ đọc CSV để không nhầm xuống dòng trong văn bản có dấu nháy với bản ghi mới.
`dòng dư` là số lần xuất hiện sau lần đầu (`duplicated()`);
`dòng liên quan` gồm tất cả dòng thuộc nhóm lặp (`duplicated(keep=False)`).
Chuỗi chỉ chứa khoảng trắng được tính riêng, sau đó gộp vào mặt nạ thiếu để kiểm tra.

In [ ]:
empty_records = 0
all_blank_records = 0
wrong_width_records = 0
csv_records = 0
with RAW_PATH.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        csv_records += 1
        if not row:
            empty_records += 1
        else:
            all_blank_records += all(not value.strip() for value in row)
            wrong_width_records += len(row) != len(header)

missing = movies.isna().copy()
space_only = pd.Series(0, index=movies.columns, dtype='int64')
for col in movies.select_dtypes(include=['object', 'string']):
    blank = movies[col].str.fullmatch(r'\s+', na=False)
    space_only[col] = blank.sum()
    missing[col] |= blank

structure = pd.Series({
    'bản ghi CSV kể cả bản ghi rỗng': csv_records,
    'bản ghi rỗng không trường': empty_records,
    'bản ghi mọi trường trống/khoảng trắng': all_blank_records,
    'bản ghi không rỗng sai số trường': wrong_width_records,
    'dòng mọi cột thiếu sau đọc': int(missing.all(axis=1).sum()),
    'dòng trùng toàn bộ (dư)': int(movies.duplicated().sum()),
})
display(structure.to_frame('Số lượng'))
key_rows = []
for col in ['id', 'tconst']:
    known = movies.loc[~missing[col], col]
    key_rows.append([col, missing[col].sum(), known.nunique(), known.duplicated().sum(),
                     known.duplicated(keep=False).sum()])
key_audit = pd.DataFrame(key_rows, columns=['khóa', 'thiếu', 'giá trị khác nhau', 'dòng dư', 'dòng liên quan'])
display(key_audit)
id_to_imdb = movies.groupby('id')['tconst'].nunique()
imdb_to_id = movies.groupby('tconst')['id'].nunique()
display(pd.Series({'id có nhiều tconst': int(id_to_imdb.gt(1).sum()),
                   'tconst có nhiều id': int(imdb_to_id.gt(1).sum()),
                   'id không dương': int(movies.id.le(0).sum()),
                   'tconst sai mẫu tt + chữ số': int((~movies.tconst.str.fullmatch(r'tt\d+', na=False)).sum())}))
dup_id_rows = movies.loc[movies.id.duplicated(keep=False)]
conflict_counts = dup_id_rows.groupby('id').nunique(dropna=False).gt(1).sum().sort_values(ascending=False)
display(conflict_counts.to_frame('Số nhóm id có nhiều giá trị kể cả thiếu'))
display(movies.groupby('tconst').agg(so_dong=('id', 'size'), so_id=('id', 'nunique'))
        .sort_values('so_dong', ascending=False).head(10))
unique_keys = (~movies.id.duplicated(keep=False) & ~movies.tconst.duplicated(keep=False)
               & ~missing.id & ~missing.tconst)
display(Markdown(f'Có **{movies.id.nunique():,} id TMDB** và **{movies.tconst.nunique():,} tconst IMDb** '
                 f'trong **{n:,} bản ghi**. Không thể coi từng dòng là một phim độc lập. '
                 f'Mẫu có cả hai mã xuất hiện đúng một lần gồm **{unique_keys.sum():,} dòng** '
                 f'({unique_keys.mean():.2%} raw); người dùng đã chọn mẫu A này cho phần quan hệ.'))

### Ý nghĩa và hướng sử dụng 29 cột

Kiểu dưới đây là kiểu pandas đọc được; ý nghĩa lấy từ README và tài liệu nguồn ở mục 1.
Các hướng xử lý chỉ là đề xuất cho bước phân tích. Chưa bỏ cột hoặc thay thế giá trị trong `movies`.

In [ ]:
column_notes = {
    'id': ('Mã tác phẩm TMDB', 'Giữ làm khóa, kiểm tra lặp; không dùng như đại lượng số'),
    'title': ('Tên tác phẩm', 'Giữ để kiểm tra ví dụ, không dùng tên làm khóa'),
    'vote_average': ('Điểm TMDB trên thang 0–10', 'Xét cùng vote_count; đánh dấu điểm 0 cần kiểm tra'),
    'vote_count': ('Số lượt chấm TMDB', 'Giữ số nguyên không âm; kiểm tra các ngưỡng lượt chấm'),
    'status': ('Trạng thái sản xuất/phát hành', 'Có thể chuyển category; giữ để phân tầng'),
    'release_date': ('Ngày phát hành được ghi nhận', 'Tạo datetime với lỗi thành NaT; tách năm/thập niên'),
    'revenue': ('Doanh thu được ghi nhận; đơn vị từng dòng chưa xác minh', 'Đếm riêng 0; không suy ra doanh thu thực bằng 0'),
    'runtime': ('Thời lượng theo phút', 'Kiểm tra 0 và cực trị, không tự cắt ngưỡng'),
    'adult': ('Cờ nội dung người lớn TMDB', 'Giữ boolean, có thể phân tầng; không tự loại'),
    'backdrop_path': ('Đường dẫn ảnh nền', 'Bỏ khỏi ma trận phân tích số nếu không dùng ảnh'),
    'budget': ('Ngân sách được ghi nhận; đơn vị từng dòng chưa xác minh', 'Đếm riêng 0; không tự điền bằng trung bình'),
    'homepage': ('Trang web của tác phẩm', 'Bỏ khỏi mô hình nếu không dùng; kiểm tra ô khoảng trắng'),
    'tconst': ('Mã tác phẩm IMDb', 'Giữ chuỗi, kiểm tra lặp và quan hệ với id'),
    'original_language': ('Mã ngôn ngữ gốc', 'Kiểm tra nhãn/nhóm hiếm trước khi chuyển category'),
    'original_title': ('Tên nguyên gốc', 'Giữ để đối chiếu; không suy ra khóa duy nhất'),
    'overview': ('Tóm tắt nội dung', 'Có thể bỏ ở phân tích dạng bảng; không tự điền văn bản'),
    'popularity': ('Chỉ số phổ biến TMDB tại thời điểm nguồn', 'Giữ số; không coi là doanh thu hay điểm đánh giá'),
    'poster_path': ('Đường dẫn poster', 'Bỏ khỏi ma trận phân tích số nếu không dùng ảnh'),
    'tagline': ('Câu giới thiệu', 'Có thể bỏ khi không làm NLP'),
    'genres': ('Các thể loại, nhiều nhãn trên một dòng', 'Tách dấu phẩy sau kiểm tra; tránh cộng trùng doanh thu'),
    'production_companies': ('Các công ty sản xuất', 'Đa nhãn; kiểm tra dấu phân cách trước khi tách'),
    'production_countries': ('Các quốc gia sản xuất', 'Đa nhãn; giữ thông tin thiếu'),
    'spoken_languages': ('Các ngôn ngữ được sử dụng', 'Đa nhãn, khác ngôn ngữ gốc'),
    'keywords': ('Từ khóa nội dung', 'Đa nhãn; có thể bỏ nếu ngoài hướng câu hỏi'),
    'directors': ('Tên đạo diễn', 'Có thể nhiều người; tên không bảo đảm định danh duy nhất'),
    'writers': ('Tên biên kịch', 'Có thể nhiều người; không tự gộp người trùng tên'),
    'averageRating': ('Điểm trung bình có trọng số IMDb, thang 1–10', 'Xét cùng numVotes'),
    'numVotes': ('Số lượt chấm IMDb', 'Giữ số nguyên không âm; không nhầm với vote_count'),
    'cast': ('Danh sách diễn viên', 'Kiểm tra khác biệt giữa các dòng cùng mã; chưa gộp tự động'),
}
assert set(column_notes) == set(movies.columns), 'Schema CSV đã thay đổi: cập nhật từ điển cột.'
schema = pd.DataFrame(column_notes, index=['ý nghĩa', 'hướng xử lý dự kiến']).T
schema.insert(0, 'dtype raw', movies.dtypes.astype(str))
schema.insert(1, 'số giá trị khác nhau', movies.nunique(dropna=True))
with pd.option_context('display.max_colwidth', None):
    display(schema)

## 3. Dữ liệu thiếu và sự thiếu đồng thời

Mẫu số các tỷ lệ là toàn bộ bản ghi raw. `Thiếu` gồm ô trống, ký hiệu `\N` và chuỗi chỉ có khoảng trắng.
Số **0** của tài chính/thời lượng được báo cáo riêng, chưa chuyển thành thiếu.
Sự thiếu đồng thời hoặc khác biệt giữa các nhóm chưa chứng minh nguyên nhân thiếu (MCAR/MAR/MNAR).

In [ ]:
missing_table = pd.DataFrame({'NA khi đọc': movies.isna().sum(), 'chỉ khoảng trắng': space_only,
                              'số thiếu': missing.sum(), 'tỷ lệ thiếu (%)': 100 * missing.mean()})
missing_table = missing_table.sort_values('số thiếu', ascending=False)
display(missing_table.round(2))
fig, ax = plt.subplots(figsize=(9, 5.5), layout='constrained')
shown = missing_table.loc[missing_table['số thiếu'].gt(0)].sort_values('tỷ lệ thiếu (%)')
ax.barh(shown.index, shown['tỷ lệ thiếu (%)'], color='#287e8e')
ax.set(xlabel='Tỷ lệ thiếu trong bản ghi raw (%)', ylabel='Cột',
       title=f'Dữ liệu thiếu theo cột — n = {n:,} bản ghi', xlim=(0, 100))
plt.show()
important = ['release_date', 'genres', 'production_companies', 'production_countries',
             'directors', 'writers', 'cast']
patterns = missing[important].value_counts().head(10).rename('số dòng').reset_index()
patterns['tỷ lệ raw (%)'] = 100 * patterns['số dòng'] / n
display(patterns.round(2))
display((missing[important].astype('int64').T @ missing[important].astype('int64'))
        .rename_axis(index='Thiếu ở cột', columns='Đồng thời thiếu ở cột'))
# Cờ 0 là chẩn đoán riêng, không được trộn vào định nghĩa thiếu thật.
availability = missing[important].rename(columns=lambda c: 'thiếu_' + c).copy()
availability['budget_bằng_0'] = movies.budget.eq(0)
availability['revenue_bằng_0'] = movies.revenue.eq(0)
availability['runtime_bằng_0'] = movies.runtime.eq(0)
availability['TMDB_không_lượt_chấm'] = movies.vote_count.eq(0)

def missing_by_group(groups):
    result = availability.groupby(groups, observed=True, dropna=False).mean().mul(100)
    result.insert(0, 'số dòng', groups.value_counts(dropna=False))
    return result.round(2)

display(missing_by_group(movies.status))
top_languages = movies.original_language.value_counts().head(10).index
language_groups = movies.original_language.where(movies.original_language.isin(top_languages), 'Các ngôn ngữ còn lại')
display(missing_by_group(language_groups))
release_dates = pd.to_datetime(movies.release_date, format='%Y-%m-%d', errors='coerce')
decades = ((release_dates.dt.year // 10) * 10).astype('Int64').astype('string').fillna('Thiếu/không đọc được')
display(missing_by_group(decades))

**Hướng xử lý dự kiến:** giữ riêng cờ thiếu; chỉ lọc các trường cần thiết cho từng câu hỏi và báo cáo số mẫu.
Không bỏ mọi dòng chỉ vì thiếu ảnh, homepage hoặc tagline. Ngày thiếu không được tự suy từ tên phim.
Thể loại/đạo diễn thiếu có thể thành nhóm “chưa có thông tin” khi so sánh, nhưng cần kiểm tra thiên lệch nhóm.
Chưa xác minh vì sao từng ô bị thiếu; không gán nguyên nhân do ngân sách thấp hoặc phim ít nổi tiếng chỉ từ các bảng này.

## 4. Phân phối số, số 0 và cực trị

Bảng đầu giữ toàn bộ giá trị raw. Bảng thứ hai chỉ mô tả các giá trị **dương**, có báo cáo số mẫu;
đây là chẩn đoán sự lệch phân phối, chưa phải quyết định loại số 0.
Ngưỡng Tukey `Q1 − 1,5 × IQR`, `Q3 + 1,5 × IQR` chỉ gắn cờ để kiểm tra.
Giá trị ngoài ngưỡng chưa đủ để kết luận sai hay tự loại phim.

In [ ]:
numeric_cols = ['budget', 'revenue', 'runtime', 'vote_average', 'vote_count',
                'averageRating', 'numVotes', 'popularity']
numeric_summary = movies[numeric_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T
numeric_summary['IQR'] = numeric_summary['75%'] - numeric_summary['25%']
numeric_summary['số 0'] = movies[numeric_cols].eq(0).sum()
numeric_summary['tỷ lệ 0 (%)'] = 100 * movies[numeric_cols].eq(0).mean()
numeric_summary['số âm'] = movies[numeric_cols].lt(0).sum()
with pd.option_context('display.max_columns', None):
    display(numeric_summary.round(3))

positive_cols = ['budget', 'revenue', 'runtime', 'vote_count', 'numVotes', 'popularity']
positive_summary = pd.DataFrame({col: movies.loc[movies[col].gt(0), col]
    .describe(percentiles=[.01, .25, .5, .75, .99]) for col in positive_cols}).T
display(positive_summary.round(3))

outliers = []
for col in numeric_cols:
    s = movies.loc[movies[col].gt(0), col] if col in positive_cols else movies[col].dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = s.lt(lo) | s.gt(hi)
    outliers.append([col, '> 0' if col in positive_cols else 'không thiếu', len(s),
                     lo, hi, flagged.sum(), 100 * flagged.mean()])
display(pd.DataFrame(outliers, columns=['cột', 'mẫu', 'n', 'ngưỡng dưới', 'ngưỡng trên',
                                        'số cờ IQR', 'tỷ lệ cờ (%)']).round(3))

numeric_checks = pd.Series({
    'budget hoặc revenue âm': int((movies.budget.lt(0) | movies.revenue.lt(0)).sum()),
    'budget và revenue cùng dương': int((movies.budget.gt(0) & movies.revenue.gt(0)).sum()),
    'TMDB ngoài [0, 10]': int((~movies.vote_average.between(0, 10)).sum()),
    'IMDb ngoài [1, 10]': int((~movies.averageRating.between(1, 10)).sum()),
    'TMDB điểm 0 nhưng có lượt chấm': int((movies.vote_average.eq(0) & movies.vote_count.gt(0)).sum()),
    'TMDB điểm dương nhưng 0 lượt chấm': int((movies.vote_average.gt(0) & movies.vote_count.eq(0)).sum()),
    'runtime bằng 0': int(movies.runtime.eq(0).sum()),
    'runtime trên 600 phút (cờ kiểm tra, không loại)': int(movies.runtime.gt(600).sum()),
})
display(numeric_checks.to_frame('Số bản ghi'))
for col in ['budget', 'revenue', 'runtime']:
    display(Markdown(f'**Ví dụ cực trị `{col}`: 3 dòng lớn nhất và 3 dòng dương nhỏ nhất.**'))
    positive = movies.loc[movies[col].gt(0)]
    example_index = positive.nlargest(3, col).index.union(positive.nsmallest(3, col).index)
    display(movies.loc[example_index, ['id', 'tconst', 'title', 'release_date', col]]
            .sort_values(col, ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5), layout='constrained')
for ax, col in zip(axes.flat, numeric_cols):
    s = movies[col].dropna()
    if col in positive_cols:
        s = s[s.gt(0)]
        plotted = np.log10(s)
        label = f'log10({col})'
        scope = '> 0'
    else:
        plotted = s
        label = f'{col} (điểm)'
        scope = 'raw'
    ax.hist(plotted, bins=45, color='#287e8e', edgecolor='white', linewidth=.25)
    ax.set(title=f'{col}: {scope}, n = {len(s):,}', xlabel=label, ylabel='Số bản ghi')
fig.suptitle('Phân phối raw và phần dương; chưa khử trùng theo phim', fontsize=14)
plt.show()

**Cách đọc:** dùng log10 cho các biến dương lệch phải để vẫn thấy toàn bộ khoảng giá trị.
Các số 0 không hiện trong histogram log, nhưng đã được đếm ở bảng phía trên.
`runtime` có đơn vị phút; tài chính chưa có metadata xác nhận đơn vị từng dòng.
Ngưỡng 600 phút chỉ giúp tìm ví dụ cần đối chiếu, không khẳng định tác phẩm dài là lỗi.
Không winsorize, điền thiếu, điều chỉnh lạm phát hoặc suy ra lợi nhuận trong notebook này.

## 5. Biến phân loại, đa nhãn và ngày

Đếm nhóm hiếm ở mức **tối đa 5 bản ghi** chỉ để chẩn đoán, chưa gom vào “khác” cho phân tích.
Kiểm tra nhãn sau `strip()` và `casefold()` để tìm các cách viết có thể trùng nghĩa.
Không tự gộp nhãn nếu chưa xác minh ý nghĩa.

In [ ]:
category_audit = []
for col in ['original_language', 'status', 'adult']:
    counts = movies.loc[~missing[col], col].value_counts()
    category_audit.append([col, len(counts), int(counts.le(5).sum()), int(counts[counts.le(5)].sum()),
                           int(missing[col].sum())])
    display(Markdown(f'**`{col}`: nhóm phổ biến (tối đa 15).**'))
    display(pd.DataFrame({'số bản ghi': counts.head(15), 'tỷ lệ raw (%)': counts.head(15) * 100 / n}).round(2))
display(pd.DataFrame(category_audit, columns=['cột', 'số nhãn', 'nhãn có ≤5 dòng',
                                             'dòng thuộc nhãn hiếm', 'số thiếu']))
label_checks = []
for col in ['original_language', 'status']:
    values = movies.loc[~missing[col], col]
    pairs = pd.DataFrame({'raw': values, 'normalized': values.str.strip().str.casefold()}).drop_duplicates()
    collisions = pairs.groupby('normalized').raw.nunique()
    label_checks.append([col, int(values.ne(values.str.strip()).sum()), int(collisions.gt(1).sum())])
    rare = values.value_counts()
    display(rare[rare.le(5)].head(20).to_frame(f'{col}: ví dụ nhãn hiếm'))
display(pd.DataFrame(label_checks, columns=['cột', 'dòng có khoảng trắng thừa', 'nhãn chuẩn hóa có nhiều cách viết']))
display(pd.Series({'original_language không phải 2 chữ cái thường':
    int((~movies.original_language.str.fullmatch(r'[a-z]{2}', na=False)).sum())}))

In [ ]:
display(movies.loc[~missing.genres, 'genres'].head(10).to_frame('Ví dụ chuỗi genres raw'))
# Chỉ áp dụng parser này cho genres: dữ liệu thực dùng tên thể loại phân cách bằng dấu phẩy.
genre_lists = movies.genres.fillna('').str.split(',').map(lambda parts: [p.strip() for p in parts if p.strip()])
genre_count = genre_lists.map(len)
genre_duplicate_tokens = genre_lists.map(lambda parts: len(parts) != len(set(parts)))
genre_brackets = movies.genres.fillna('').str.contains(r'[\[\]{}]', regex=True)
display(pd.Series({'dòng thiếu genres': int(missing.genres.sum()),
    'dòng có nhiều thể loại': int(genre_count.gt(1).sum()),
    'dòng lặp nhãn trong cùng genres': int(genre_duplicate_tokens.sum()),
    'dòng chứa dấu ngoặc cần parser khác': int(genre_brackets.sum()),
    'số thể loại tối đa trên một dòng': int(genre_count.max())}))
genre_tokens = genre_lists.map(lambda parts: sorted(set(parts))).explode().dropna()
genre_counts = genre_tokens.value_counts()
display(pd.DataFrame({'số bản ghi chứa thể loại': genre_counts,
                      'tỷ lệ trong raw (%)': 100 * genre_counts / n}).round(2))
display(genre_count.value_counts().sort_index().to_frame('Số bản ghi theo số nhãn genres'))

invalid_dates = ~missing.release_date & release_dates.isna()
non_iso = ~missing.release_date & ~movies.release_date.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)
date_audit = pd.Series({'thiếu release_date': int(missing.release_date.sum()),
    'có giá trị nhưng không parse được': int(invalid_dates.sum()),
    'chuỗi không đúng YYYY-MM-DD': int(non_iso.sum()),
    'ngày sớm nhất': str(release_dates.min().date()),
    'ngày muộn nhất': str(release_dates.max().date()),
    'sau ngày nhóm tải 05/11/2026': int(release_dates.gt(DOWNLOAD_DATE).sum()),
    'Released nhưng ngày sau ngày tải': int((release_dates.gt(DOWNLOAD_DATE) & movies.status.eq('Released')).sum())})
display(date_audit.to_frame('Kết quả'))
display(movies.loc[invalid_dates | non_iso | release_dates.gt(DOWNLOAD_DATE),
                   ['id', 'tconst', 'title', 'release_date', 'status']].head(15))

Mỗi phim có thể thuộc nhiều thể loại nên tổng tỷ lệ thể loại có thể vượt 100%.
Bảng trên đếm số **bản ghi raw chứa nhãn**, chưa là số phim duy nhất.
Không cộng doanh thu sau khi `explode()` nếu chưa quy định cách phân bổ.
Không tái dùng cách tách `genres` cho tên công ty/người khi chưa kiểm tra vì tên có thể chứa dấu phẩy.
Ngày tương lai có thể là lịch dự kiến; ngày hợp lệ về cú pháp không bảo đảm đúng về nội dung.
Các dòng `Released` nhưng ngày sau ngày tải cần được đối chiếu riêng với nguồn.

## 6. Mối liên hệ sơ bộ — đã chọn mẫu A

**Quyết định ngày 15/11/2026:** người dùng chọn **A**. Giữ raw để kiểm tra chất lượng,
chỉ dùng dòng có cả `id` và `tconst` xuất hiện đúng một lần cho quan hệ sơ bộ.
Các phương án đã cân nhắc:

| Chọn | Cách làm | Ưu điểm | Nhược điểm |
| --- | --- | --- | --- |
| A (đề xuất) | Raw cho kiểm tra chất lượng; quan hệ trên dòng có cả `id` và `tconst` xuất hiện đúng một lần | Không đếm lặp, chưa đoán cách gộp | Loại cả phim chỉ khác danh sách diễn viên; có thể lệch mẫu |
| B | Gộp cùng `id`, kiểm tra xung đột trước khi tính quan hệ | Giữ được nhiều phim | Cần thêm quy tắc chọn giá trị và xử lý `tconst` dùng chung |
| C | Tính trên toàn bộ bản ghi raw | Ít biến đổi dữ liệu | Phim có nhiều dòng mang trọng số lớn hơn; không suy ra quan hệ ở mức phim |

**Quyết định 2 — A:** quan hệ tài chính chỉ dùng `budget > 0` và `revenue > 0`.
Ưu điểm: tránh diễn giải các số 0 chưa rõ nghĩa. Nhược điểm: mẫu nhỏ và có thể thiên lệch do độ đầy đủ dữ liệu.

**Quyết định 3 — A:** so sánh điểm trên toàn bộ mẫu hợp lệ có lượt chấm,
rồi kiểm tra thêm các ngưỡng **10, 100, 1.000 lượt ở cả hai nền tảng**.
Ưu điểm: thấy độ nhạy với ngưỡng. Nhược điểm: các mẫu lồng nhau và khác nhau về thành phần phim.
Các lựa chọn chỉ áp dụng cho phần khám phá này, giữ nguyên CSV gốc.

In [ ]:
analysis_base = movies.loc[unique_keys].copy()
sample_flow = pd.DataFrame([
    ['Raw', n, 0],
    ['Cả id và tconst xuất hiện đúng một lần', len(analysis_base), n - len(analysis_base)],
], columns=['bước', 'số dòng còn lại', 'số dòng loại ở bước này'])
sample_flow['tỷ lệ giữ so với raw (%)'] = 100 * sample_flow['số dòng còn lại'] / n
display(sample_flow.round(2))
assert analysis_base.id.is_unique and analysis_base.tconst.is_unique
comparison = pd.DataFrame({label: {
    'số dòng': len(frame),
    'budget > 0 (%)': 100 * frame.budget.gt(0).mean(),
    'revenue > 0 (%)': 100 * frame.revenue.gt(0).mean(),
    'thiếu genres (%)': 100 * frame.genres.isna().mean(),
    'ngôn ngữ gốc en (%)': 100 * frame.original_language.eq('en').mean(),
    'status Released (%)': 100 * frame.status.eq('Released').mean(),
} for label, frame in [('Raw', movies), ('Mẫu A', analysis_base)]})
display(comparison.round(2))

### 6.1. Ngân sách và doanh thu được ghi nhận

Từ mẫu A, chỉ giữ các dòng có **cả `budget > 0` và `revenue > 0`** theo quyết định 2A.
Giữ toàn bộ cực trị dương và các trạng thái trong mẫu này, chưa áp dụng ngưỡng ngân sách, năm hay outlier khác.
So sánh Pearson trên giá trị gốc, Pearson sau log10 và Spearman theo thứ hạng để mô tả các mặt khác nhau của quan hệ.
Log10 chỉ dùng để hiển thị và tính thêm thống kê, không sửa các cột gốc.

In [ ]:
finance = analysis_base.loc[analysis_base.budget.gt(0) & analysis_base.revenue.gt(0)].copy()
finance_flow = pd.DataFrame([
    ['Mẫu A', len(analysis_base), 0],
    ['budget > 0 và revenue > 0', len(finance), len(analysis_base) - len(finance)],
], columns=['bước', 'số dòng còn lại', 'số dòng loại ở bước này'])
finance_flow['tỷ lệ so với mẫu A (%)'] = 100 * finance_flow['số dòng còn lại'] / len(analysis_base)
finance_flow['tỷ lệ so với raw (%)'] = 100 * finance_flow['số dòng còn lại'] / n
display(finance_flow.round(2))
finance_log = np.log10(finance[['budget', 'revenue']])
finance_metrics = pd.Series({
    'n': len(finance),
    'Pearson giá trị gốc': finance.budget.corr(finance.revenue, method='pearson'),
    'Pearson log10': finance_log.budget.corr(finance_log.revenue, method='pearson'),
    'Spearman': finance.budget.corr(finance.revenue, method='spearman'),
}, name='budget – revenue')
display(finance_metrics.to_frame().round(4))
display(finance[['budget', 'revenue']].describe(percentiles=[.25, .5, .75, .99]).T.round(2))
finance_profile = pd.DataFrame({label: {
    'số dòng': len(frame),
    'ngôn ngữ gốc en (%)': 100 * frame.original_language.eq('en').mean(),
    'status Released (%)': 100 * frame.status.eq('Released').mean(),
    'năm phát hành trung vị': release_dates.loc[frame.index].dt.year.median(),
} for label, frame in [('Mẫu A', analysis_base), ('Tài chính dương', finance)]})
display(finance_profile.round(2))

fig, ax = plt.subplots(figsize=(8, 5.5), layout='constrained')
hb = ax.hexbin(finance_log.budget, finance_log.revenue, gridsize=45,
               mincnt=1, bins='log', cmap='viridis')
fig.colorbar(hb, ax=ax, label='Số dòng trong ô lục giác (thang log)')
ax.set(title=f'Ngân sách và doanh thu dương — n = {len(finance):,}',
       xlabel='log10(budget, giá trị CSV)', ylabel='log10(revenue, giá trị CSV)')
plt.show()
display(Markdown(f'Mẫu tài chính có **{len(finance):,} dòng** '
    f'(**{len(finance)/len(analysis_base):.2%}** mẫu A; **{len(finance)/n:.2%}** raw). '
    f'Pearson gốc = **{finance_metrics["Pearson giá trị gốc"]:.3f}**, '
    f'Pearson log10 = **{finance_metrics["Pearson log10"]:.3f}**, '
    f'Spearman = **{finance_metrics["Spearman"]:.3f}**. '
    'Các hệ số mô tả quan hệ trong mẫu đã lọc; chưa chứng minh ngân sách gây ra doanh thu.'))

Biểu đồ dùng **toàn bộ mẫu tài chính**, gộp mật độ vào các ô lục giác để giảm chồng điểm, không lấy mẫu ngẫu nhiên.
Đối chiếu bảng thành phần với mẫu A trước khi khái quát kết quả. Mẫu có tài chính dương có thể khác các dòng bị loại.
Chưa xác minh đơn vị tiền tệ từng dòng, chưa điều chỉnh thời gian/lạm phát và chưa kiểm soát thể loại.
Hệ số trên log10 mô tả quan hệ trên thang biến đổi, không so trực tiếp độ lớn với hệ số trên giá trị gốc.

### 6.2. Điểm TMDB–IMDb và lượt chấm

Từ mẫu A, mẫu so sánh cặp gồm **TMDB trong [0, 10], IMDb trong [1, 10], `vote_count > 0` và `numVotes > 0`**.
Điểm TMDB bằng 0 nhưng có lượt vẫn nằm trong miền giá trị được kiểm tra; số trường hợp này được báo cáo riêng,
chưa tự sửa hoặc loại thêm. Đặt `Δ = vote_average − averageRating` (điểm).

Các ngưỡng 10/100/1.000 áp dụng đồng thời cho `vote_count` và `numVotes`, bắt đầu từ cùng mẫu cặp hợp lệ.
Đây là các mẫu **lồng nhau**, không phải bốn nhóm phim độc lập.
So sánh Pearson, Spearman, chênh lệch trung bình/trung vị và chênh lệch tuyệt đối trung bình.
Chênh lệch dương nghĩa là điểm TMDB cao hơn IMDb trong mẫu đó.

In [ ]:
rating_mask = (analysis_base.vote_average.between(0, 10)
               & analysis_base.averageRating.between(1, 10)
               & analysis_base.vote_count.gt(0) & analysis_base.numVotes.gt(0))
paired_ratings = analysis_base.loc[rating_mask].copy()
paired_ratings['rating_gap'] = paired_ratings.vote_average - paired_ratings.averageRating
rating_rows = []
threshold_samples = {}
for threshold in [1, 10, 100, 1000]:
    subset = paired_ratings.loc[paired_ratings.vote_count.ge(threshold)
                                 & paired_ratings.numVotes.ge(threshold)]
    threshold_samples[threshold] = subset
    rating_rows.append({
        'ngưỡng lượt ở cả hai': threshold,
        'n': len(subset),
        'loại thêm từ mẫu cặp hợp lệ': len(paired_ratings) - len(subset),
        'giữ so với mẫu cặp (%)': 100 * len(subset) / len(paired_ratings),
        'Pearson': subset.vote_average.corr(subset.averageRating),
        'Spearman': subset.vote_average.corr(subset.averageRating, method='spearman'),
        'Δ trung bình': subset.rating_gap.mean(),
        'Δ trung vị': subset.rating_gap.median(),
        '|Δ| trung bình': subset.rating_gap.abs().mean(),
        'TMDB điểm 0 có lượt': int(subset.vote_average.eq(0).sum()),
    })
rating_sensitivity = pd.DataFrame(rating_rows).set_index('ngưỡng lượt ở cả hai')
rating_flow = pd.DataFrame([
    ['Mẫu A', len(analysis_base), 0],
    ['Cặp điểm hợp lệ và có lượt ở cả hai nền tảng', len(paired_ratings), len(analysis_base) - len(paired_ratings)],
], columns=['bước', 'số dòng còn lại', 'số dòng loại ở bước này'])
display(rating_flow)
with pd.option_context('display.max_columns', None):
    display(rating_sensitivity.round(4))

# Quan hệ IMDb với lượt chấm dùng mẫu IMDb hợp lệ riêng, không đòi phim có lượt TMDB.
imdb_ratings = analysis_base.loc[analysis_base.averageRating.between(1, 10)
                                 & analysis_base.numVotes.gt(0)].copy()
display(pd.DataFrame([
    ['Mẫu A', len(analysis_base), 0],
    ['IMDb điểm [1, 10] và numVotes > 0', len(imdb_ratings), len(analysis_base) - len(imdb_ratings)],
], columns=['bước', 'số dòng còn lại', 'số dòng loại ở bước này']))
imdb_votes_rho = imdb_ratings.numVotes.corr(imdb_ratings.averageRating, method='spearman')
print(f'IMDb theo lượt: n = {len(imdb_ratings):,}; Spearman = {imdb_votes_rho:.4f}')
vote_groups = pd.cut(imdb_ratings.numVotes, bins=[1, 10, 100, 1000, 10000, 100000, np.inf],
                     right=False, labels=['1–9', '10–99', '100–999', '1.000–9.999',
                                          '10.000–99.999', '≥100.000'])
imdb_by_votes = imdb_ratings.groupby(vote_groups, observed=True).agg(
    n=('id', 'size'), diem_trung_binh=('averageRating', 'mean'),
    diem_trung_vi=('averageRating', 'median'), do_lech_chuan=('averageRating', 'std'))
imdb_by_votes.index.name = 'Nhóm lượt chấm IMDb (không chồng lấn)'
display(imdb_by_votes.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), layout='constrained')
hb = axes[0].hexbin(paired_ratings.vote_average, paired_ratings.averageRating,
                    gridsize=45, mincnt=1, bins='log', cmap='viridis', extent=(0, 10, 0, 10))
axes[0].plot([0, 10], [0, 10], '--', color='#d35400', linewidth=1.5, label='Hai điểm bằng nhau')
axes[0].set(title=f'TMDB và IMDb — n = {len(paired_ratings):,}',
            xlabel='Điểm TMDB (0–10)', ylabel='Điểm IMDb (1–10)', xlim=(0, 10), ylim=(0, 10))
axes[0].legend(loc='upper left', fontsize=8)
fig.colorbar(hb, ax=axes[0], label='Số dòng trong ô (thang log)')
hb = axes[1].hexbin(np.log10(imdb_ratings.numVotes), imdb_ratings.averageRating,
                    gridsize=45, mincnt=1, bins='log', cmap='viridis')
axes[1].set(title=f'IMDb theo lượt chấm — n = {len(imdb_ratings):,}',
            xlabel='log10(numVotes IMDb)', ylabel='Điểm IMDb (1–10)', ylim=(0.8, 10.2))
fig.colorbar(hb, ax=axes[1], label='Số dòng trong ô (thang log)')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.6), layout='constrained')
positions = np.arange(len(rating_sensitivity))
ax.plot(positions, rating_sensitivity['Δ trung bình'], 'o-', label='Δ trung bình')
ax.plot(positions, rating_sensitivity['Δ trung vị'], 's--', label='Δ trung vị')
ax.axhline(0, color='grey', linewidth=1)
ax.set_xticks(positions, [f'≥{threshold:,}\nn = {int(row["n"]):,}'
                        for threshold, row in rating_sensitivity.iterrows()])
ax.set(title='Chênh lệch điểm khi thay đổi ngưỡng lượt chấm',
       xlabel='Số lượt tối thiểu ở cả hai nền tảng (các mẫu lồng nhau)',
       ylabel='Δ = TMDB − IMDb (điểm)')
ax.legend()
plt.show()
display(Markdown(f'Mẫu cặp hợp lệ gồm **{len(paired_ratings):,} dòng** '
    f'({len(paired_ratings)/len(analysis_base):.2%} mẫu A). '
    f'Chênh lệch trung bình Δ = **{rating_sensitivity.loc[1, "Δ trung bình"]:.3f}** điểm '
    f'và Spearman = **{rating_sensitivity.loc[1, "Spearman"]:.3f}**. '
    f'Ở ngưỡng ≥1.000 lượt cả hai, còn **{int(rating_sensitivity.loc[1000, "n"]):,} dòng**, '
    f'Δ trung bình = **{rating_sensitivity.loc[1000, "Δ trung bình"]:.3f}** điểm '
    f'và Spearman = **{rating_sensitivity.loc[1000, "Spearman"]:.3f}**.'))

Các biểu đồ dùng toàn bộ mẫu được nêu, không lấy ngẫu nhiên một phần phim.
Tương quan đo mức đồng biến, không đo hai nền tảng cho điểm bằng nhau; xem thêm `Δ` và `|Δ|`.
Thay đổi khi nâng ngưỡng có thể phản ánh thành phần phim và khán giả khác nhau,
không chứng minh nhiều lượt chấm làm điểm tăng/giảm hoặc nền tảng nào đánh giá “đúng” hơn.
Điểm hai nguồn có thể được chụp ở thời điểm khác nhau; dữ liệu không chứa lịch sử lượt chấm.
Đây là khám phá mẫu hình, chưa có kiểm định, khoảng tin cậy hay kiểm soát yếu tố gây nhiễu.

## 7. Tóm tắt khám phá và hướng tiền xử lý

Các số liệu dưới đây được tính từ kết quả chạy; nhóm dùng để tự viết nhận xét của mình.
Phạm vi là các bản ghi raw và những mẫu con đã nêu, chưa chốt bốn câu hỏi phân tích của nhóm.

In [ ]:
finance_positive = movies.budget.gt(0) & movies.revenue.gt(0)
notes = [
    f'CSV có {n:,} bản ghi, {movies.shape[1]} cột, {int(movies.duplicated().sum()):,} dòng trùng toàn bộ; '
    f'nhưng {int(movies.id.duplicated().sum()):,} dòng dư theo id và {int(movies.tconst.duplicated().sum()):,} dòng dư theo tconst.',
    f'budget bằng 0 ở {int(movies.budget.eq(0).sum()):,} dòng ({movies.budget.eq(0).mean():.2%}); '
    f'revenue bằng 0 ở {int(movies.revenue.eq(0).sum()):,} dòng ({movies.revenue.eq(0).mean():.2%}). '
    f'Cả hai dương ở {int(finance_positive.sum()):,} dòng ({finance_positive.mean():.2%}).',
    f'release_date thiếu ở {int(missing.release_date.sum()):,} dòng ({missing.release_date.mean():.2%}); '
    f'genres thiếu ở {int(missing.genres.sum()):,} dòng ({missing.genres.mean():.2%}). '
    f'runtime bằng 0 ở {int(movies.runtime.eq(0).sum()):,} dòng ({movies.runtime.eq(0).mean():.2%}) '
    f'và cực đại {int(movies.runtime.max()):,} phút. Có {int(space_only.sum()):,} ô chỉ chứa khoảng trắng.',
    f'Trên {len(finance):,} dòng có hai mã duy nhất và tài chính cùng dương, '
    f'Spearman budget–revenue = {finance_metrics["Spearman"]:.3f}, '
    f'Pearson gốc = {finance_metrics["Pearson giá trị gốc"]:.3f} '
    f'và Pearson log10 = {finance_metrics["Pearson log10"]:.3f}. '
    f'Mẫu này chỉ chiếm {len(finance)/n:.2%} raw, hạn chế phạm vi khái quát.',
    f'Mẫu cặp điểm có {len(paired_ratings):,} dòng; Δ trung bình TMDB−IMDb '
    f'= {rating_sensitivity.loc[1, "Δ trung bình"]:.3f} điểm. '
    f'Khi cả hai đạt ≥1.000 lượt, còn {int(rating_sensitivity.loc[1000, "n"]):,} dòng, '
    f'Δ trung bình = {rating_sensitivity.loc[1000, "Δ trung bình"]:.3f} điểm. '
    'Phải nêu ngưỡng và số mẫu khi so sánh hai nền tảng.',
]
display(Markdown('\n\n'.join(f'{i}. {note}' for i, note in enumerate(notes, 1))))

**Vấn đề chất lượng cần lưu ý:** mã lặp và mã IMDb liên kết nhiều TMDB; số 0 tài chính/thời lượng;
điểm TMDB bằng 0 nhưng có lượt và điểm dương nhưng không có lượt; 95 dòng lặp nhãn thể loại;
2 dòng `Released` có ngày sau ngày tải. Nguyên nhân từng trường hợp chưa được xác minh.

**Đã áp dụng cho khám phá:** mẫu hai mã duy nhất cho quan hệ; tài chính dương cho quan hệ tài chính;
lọc cặp điểm theo miền giá trị và lượt chấm, so sánh độ nhạy ngưỡng 1/10/100/1.000.
Các bảng chất lượng vẫn dùng raw, chưa ghi dữ liệu processed.

**Tiền xử lý dự kiến cho phân tích sau:** nhóm thống nhất đơn vị tác phẩm và cách xử lý mã lặp;
giữ cờ thiếu/tài chính 0/thời lượng 0, lọc theo từng câu hỏi; parse ngày và thể loại trên bản phân tích;
không tự điền toàn bộ cột, không loại tất cả outlier hoặc mọi dòng có NA.
Nếu dùng tài chính, xác minh tiền tệ, phạm vi doanh thu và cách so sánh qua thời gian.

**Mẫu hình để nhóm chọn câu hỏi:** quan hệ tài chính thay đổi ra sao trong các nhóm ngân sách,
thời kỳ và thể loại; chênh lệch điểm TMDB–IMDb có còn tương tự trong các nhóm lượt chấm/thể loại;
phân bố thể loại/thời lượng thay đổi thế nào khi giữ cùng phạm vi tác phẩm qua thời gian.
Đây chưa là danh sách câu hỏi đã chốt hoặc kết luận nhân quả.

**Các điểm nhóm cần thống nhất khi chốt câu hỏi:** phim điện ảnh hay gồm phim ngắn/truyền hình;
có tiếp tục dùng mẫu hai mã duy nhất hay điều tra cách gộp để giữ thêm phim;
ngưỡng lượt chấm chính cho từng câu hỏi; cách xử lý tài chính 0, cực trị và khác biệt thời gian.
Đây là các quyết định cho bước đặt câu hỏi/phân tích tiếp theo, không cản trở bàn giao khám phá raw.

In [ ]:
# Xác nhận file nguồn không đổi trong lần chạy và các bảng kiểm tra khớp số dòng.
assert file_sha256(RAW_PATH) == raw_sha256, 'CSV đã thay đổi trong lúc chạy.'
assert wrong_width_records == 0, 'Có bản ghi sai số trường, cần kiểm tra trước khi diễn giải.'
assert csv_records - empty_records == n
assert missing.shape == movies.shape
assert missing_table['số thiếu'].between(0, n).all()
assert analysis_base.id.is_unique and analysis_base.tconst.is_unique
assert finance[['budget', 'revenue']].gt(0).all().all()
assert finance.index.isin(analysis_base.index).all()
assert paired_ratings.index.isin(analysis_base.index).all()
assert rating_sensitivity['n'].is_monotonic_decreasing
assert rating_sensitivity.loc[1, 'n'] == len(paired_ratings)
assert imdb_by_votes['n'].sum() == len(imdb_ratings)
assert rating_sensitivity[['Pearson', 'Spearman']].abs().le(1).all().all()
print('Đã kiểm tra raw và các mẫu quan hệ; SHA-256 nguồn không thay đổi.')